# IP ACL Setup for Cross-Workspace Migration

**Purpose:** Configure IP whitelisting when the target workspace has IP Access Lists enabled.

---

## When Do You Need This?

IP whitelisting is required when:
1. Target workspace has **IP Access Lists** enabled
2. You're using either **SP OAuth** or **PAT** authentication
3. Connections from the source cluster are being blocked

### How to Check if IP ACLs are Enabled

In the **target workspace**:
1. Go to **Settings** → **Security** → **IP Access Lists**
2. If you see entries and the feature is enabled, whitelisting is required

---

## Two Options for IP Whitelisting

| Option | Description | Best For |
|--------|-------------|----------|
| **Automated Scripts** | Shell scripts that auto-detect IP and whitelist | CLI users, automation |
| **Manual Setup** | Run cells below to detect IP, then add manually | Interactive users |

---

## Option 1: Automated Scripts (Recommended for CLI)

Run these commands from your local terminal (not in Databricks):

### Before Migration (Step 3.5)

```bash
cd "Customer-Folder/Catalog Migration"

# Auto-detect cluster IP and whitelist on target workspace
./scripts/auto_setup_ip_acl.sh \
  --source-profile source-workspace \
  --target-profile target-workspace
```

**What it does:**
1. Deploys IP detection job to source workspace
2. Runs job to detect cluster egress IP
3. Adds IP to target workspace allowlist
4. Waits 5 minutes for propagation
5. Verifies IP is whitelisted

### After Migration (Post Step 4)

```bash
# After validating migration, remove IP from allowlist
./scripts/cleanup_ip_acl.sh \
  --target-profile target-workspace
```

**What it does:**
1. Reads stored IP from metadata
2. Removes IP from target workspace allowlist
3. Verifies removal

---

## Option 2: Manual Setup (Interactive)

Run the cells below on your **source workspace** cluster to detect the IP, then manually add it to the target workspace.

---

In [ ]:
# ============================================================================
# Cell 1: Detect Cluster Egress IP
# ============================================================================
# Run this on your SOURCE workspace cluster
# This detects the public IP that outbound connections come from
# ============================================================================

import requests

print("="*80)
print("DETECTING CLUSTER EGRESS IP")
print("="*80)
print()

cluster_ip = None

# Method 1: ipify.org (most reliable)
try:
    print("🔍 Method 1: Using ipify.org...")
    response = requests.get('https://api.ipify.org?format=json', timeout=10)
    cluster_ip = response.json()['ip']
    print(f"   ✅ Detected IP: {cluster_ip}")
except Exception as e:
    print(f"   ❌ Failed: {e}")

# Method 2: icanhazip.com (backup)
if not cluster_ip:
    try:
        print("🔍 Method 2: Using icanhazip.com...")
        response = requests.get('https://icanhazip.com', timeout=10)
        cluster_ip = response.text.strip()
        print(f"   ✅ Detected IP: {cluster_ip}")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

# Method 3: ipinfo.io (backup)
if not cluster_ip:
    try:
        print("🔍 Method 3: Using ipinfo.io...")
        response = requests.get('https://ipinfo.io/json', timeout=10)
        cluster_ip = response.json()['ip']
        print(f"   ✅ Detected IP: {cluster_ip}")
    except Exception as e:
        print(f"   ❌ Failed: {e}")

if cluster_ip:
    print()
    print("="*80)
    print("✅ CLUSTER IP DETECTED")
    print("="*80)
    print()
    print(f"   Your Cluster Egress IP: {cluster_ip}")
    print()
    print("📋 IP Range Options for Whitelisting:")
    print(f"   • Single IP (/32):   {cluster_ip}/32          (Most restrictive - recommended)")
    
    # Calculate ranges
    parts = cluster_ip.split('.')
    small_range = f"{parts[0]}.{parts[1]}.{parts[2]}.{int(parts[3]) // 16 * 16}/28"
    large_range = f"{parts[0]}.{parts[1]}.{parts[2]}.0/24"
    print(f"   • Small range (/28): {small_range}   (16 IPs)")
    print(f"   • Large range (/24): {large_range}    (256 IPs - less secure)")
    print()
else:
    print()
    print("❌ Could not detect cluster IP")
    print("   Possible causes:")
    print("   • No internet access from cluster")
    print("   • Firewall blocking outbound connections")

## Manual Whitelisting Steps

After detecting the IP above, add it to the target workspace:

### Option A: Via Databricks CLI

```bash
# Replace YOUR_IP with the detected IP from above
databricks ip-access-lists create \
  --label "source-workspace-migration" \
  --list-type ALLOW \
  --ip-addresses "YOUR_IP/32" \
  --profile target-workspace
```

### Option B: Via Target Workspace UI

1. Open **target workspace** in browser
2. Go to **Settings** → **Security** → **IP Access Lists**
3. Click **Add IP Access List**
4. Enter:
   - **Label**: `source-workspace-migration`
   - **Type**: `Allow`
   - **IP Addresses**: The IP detected above (e.g., `35.155.15.56/32`)
5. Click **Save**

### Wait for Propagation

**Important:** Wait 5 minutes for IP ACL changes to propagate before running the migration.

---

In [ ]:
# ============================================================================
# Cell 2: Save IP to UC Volume (Optional - for cleanup script)
# ============================================================================
# This saves the detected IP to a volume file so the cleanup script can
# automatically find it later.
# ============================================================================

if cluster_ip:
    import json
    from datetime import datetime
    
    # CONFIGURE THIS: Your UC volume path
    volume_base = "/Volumes/your_catalog/your_schema/dashboard_migration"
    
    print("="*80)
    print("SAVING IP METADATA TO UC VOLUME")
    print("="*80)
    print()
    
    # Create metadata
    metadata = {
        "cluster_ip": cluster_ip,
        "detected_at": datetime.now().isoformat(),
        "cluster_id": spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "unknown"),
        "workspace_url": spark.conf.get("spark.databricks.workspaceUrl", "unknown"),
        "detected_by": "Bundle_IP_ACL_Setup.ipynb",
        "suggested_ranges": {
            "single_ip": f"{cluster_ip}/32",
            "small_range": f"{'.'.join(cluster_ip.split('.')[:3])}.{int(cluster_ip.split('.')[3]) // 16 * 16}/28",
            "large_range": f"{'.'.join(cluster_ip.split('.')[:3])}.0/24"
        }
    }
    
    ip_file = f"{volume_base}/cluster_ip_metadata.json"
    print(f"Target location: {ip_file}")
    print()
    
    try:
        metadata_json = json.dumps(metadata, indent=2)
        dbutils.fs.put(ip_file, metadata_json, overwrite=True)
        print("✅ IP metadata saved successfully")
        print()
        print("Saved metadata:")
        print(metadata_json)
        print()
        print("📝 The cleanup script can now auto-detect this IP")
    except Exception as e:
        print(f"⚠️  Could not save to volume: {e}")
        print()
        print("This is OK - you can still run cleanup with --cluster-ip flag:")
        print(f"   ./scripts/cleanup_ip_acl.sh --cluster-ip {cluster_ip}")
else:
    print("❌ No cluster IP detected - run Cell 1 first")

## Cleanup After Migration

**Important:** After validating your migration, remove the IP from the target workspace allowlist.

### Option A: Automated Cleanup Script

```bash
# Run from local terminal
./scripts/cleanup_ip_acl.sh --target-profile target-workspace
```

### Option B: Manual Removal via CLI

```bash
# List current IP ACL entries
databricks ip-access-lists list --profile target-workspace

# Delete the entry (use list_id from above)
databricks ip-access-lists delete LIST_ID --profile target-workspace
```

### Option C: Manual Removal via UI

1. Open **target workspace**
2. Go to **Settings** → **Security** → **IP Access Lists**
3. Find the entry labeled `source-workspace-migration`
4. Click **Delete**

---

## Troubleshooting

### Error: "IP blocked by Databricks IP ACL"

This means the source cluster IP is not whitelisted. Run Cell 1 to detect the IP and add it to the target workspace.

### Error: Cannot detect IP

- Check cluster has internet access
- Try a different cluster (some have restricted egress)
- Check firewall/security group settings

### IP changes after cluster restart

If your cluster IP changes after restart:
- Consider whitelisting a /28 range instead of single IP
- Or re-run the setup script after each restart

---

## Script Reference

| Script | Purpose | Command |
|--------|---------|--------|
| `auto_setup_ip_acl.sh` | Detect IP and whitelist | `./scripts/auto_setup_ip_acl.sh` |
| `cleanup_ip_acl.sh` | Remove IP after migration | `./scripts/cleanup_ip_acl.sh` |

### Common Options

```bash
# Specify profiles
--source-profile PROFILE   # Source workspace CLI profile
--target-profile PROFILE   # Target workspace CLI profile

# Provide IP directly (skip auto-detection)
--cluster-ip X.X.X.X

# Test without making changes
--dry-run

# Skip confirmation prompts (cleanup only)
--force
```